### Quick start

In [29]:
%%capture
!pip install langchain openai tiktoken chromadb langchain_community

In [15]:
from getpass import getpass
import os

OPEN_AI_API_KEY = getpass("OpenAI API key: ")
print(f"{OPEN_AI_API_KEY[:3]}...{OPEN_AI_API_KEY[-4:]}")
os.environ["OPEN_AI_API_KEY"] = OPEN_AI_API_KEY

sk-...szoA


### Load documents

In [33]:
import requests
from langchain_community.document_loaders import PyPDFLoader

urls = [
    'https://arxiv.org/pdf/2306.06031v1.pdf',
    'https://arxiv.org/pdf/2306.12156v1.pdf',
    'https://arxiv.org/pdf/2306.14289v1.pdf',
    'https://arxiv.org/pdf/2305.10973v1.pdf',
    'https://arxiv.org/pdf/2306.13643v1.pdf'
]

ml_papers = []

for i, url in enumerate(urls):
    response = requests.get(url)
    filename = f"paper{i+1}.pdf"
    with open(filename, "wb") as f: # wb means write binary and is used for writing non-text files like PDFs
        f.write(response.content)
        print(f"Downloaded {filename}")
        
    loader = PyPDFLoader(filename)
    data = loader.load()
    print(f"Loaded {len(data)} pages from {filename}")
    ml_papers.extend(data)

Downloaded paper1.pdf
Loaded 7 pages from paper1.pdf
Downloaded paper2.pdf
Loaded 11 pages from paper2.pdf
Downloaded paper3.pdf
Loaded 10 pages from paper3.pdf
Downloaded paper4.pdf
Loaded 11 pages from paper4.pdf
Downloaded paper5.pdf
Loaded 18 pages from paper5.pdf


In [35]:
type(ml_papers), len(ml_papers),  ml_papers[0]

(list,
 57,
 Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-06-12T00:32:18+00:00', 'author': '', 'keywords': '', 'moddate': '2023-06-12T00:32:18+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'templateversion': 'IJCAI.2023.0', 'title': '', 'trapped': '/False', 'source': 'paper1.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='FinGPT: Open-Source Financial Large Language Models\nHongyang (Bruce) Yang1, Xiao-Yang Liu1, Christina Dan Wang2\n1Columbia University; 2New York University (Shanghai)\n{HY2500, XL2427}@columbia.edu; christina.wang@nyu.edu\nAbstract\nLarge language models (LLMs) have shown the\npotential of revolutionizing natural language pro-\ncessing tasks in diverse domains, sparking great\ninterest in finance. Accessing high-quality fi-\nnancial data is the first challenge for financial\nLLMs (FinLLMs). While propri

In [39]:
%%capture
!pip install -U langchain-text-splitters

In [41]:
# split docs
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    length_function=len
)

documents = text_splitter.split_documents(ml_papers)

In [44]:
# embeddings
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002", openai_api_key=OPEN_AI_API_KEY)
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
)
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3
    }
)

/tmp/ipykernel_59866/4235138118.py:5: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings(model="text-embedding-ada-002", openai_api_key=OPEN_AI_API_KEY)


In [48]:
%%capture
!pip install -U langchain-openai

In [55]:
# chat model
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


chat = ChatOpenAI(
    openai_api_key=OPEN_AI_API_KEY,
    model="gpt-4o-mini",
    temperature=0.2,
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that answers questions about machine learning research papers."),
    ("human", "{query}")
])

chain = prompt_template | chat


In [63]:
query1 = "what is fingpt?"
res1 = chain.invoke({"query": query1})

query2 = "what's makes findgpt complicate to train?"
res2 = chain.invoke({"query": query2})

query3 = "what is fast segment?"
res3 = chain.invoke({"query": query3})

results = {
    0: {
        "query": query1,
        "response": res1
    },
    1: {
        "query": query2,
        "response": res2
    },
    2: {
        "query": query3,
        "response": res3
    }
}

In [67]:
results[2]["response"]

AIMessage(content='"Fast Segment" typically refers to a method or approach in the field of machine learning or computer vision that focuses on efficient segmentation of images or data. While the term itself may not refer to a specific, widely recognized algorithm or paper, it often implies techniques that aim to achieve rapid segmentation results, possibly leveraging advancements in deep learning, optimization, or computational efficiency.\n\nIn the context of image segmentation, fast segmentation methods are crucial for applications that require real-time processing, such as autonomous driving, video analysis, or interactive applications. These methods may utilize lightweight neural network architectures, optimized algorithms, or parallel processing techniques to enhance speed without significantly sacrificing accuracy.\n\nIf you have a specific paper or context in mind regarding "Fast Segment," please provide more details, and I can give you a more tailored explanation!', additional_